In [1]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
from transformers import AutoTokenizer


nltk.download("punkt")
nltk.download("punkt_tab")

text = (
    "I really enjoyed this movie. "
    "The actors were amazing, and the story was surprisingly touching! "
    "I would definitely recommend it to my friends."
)

/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/parkchanryong/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/parkchanryong/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
print("원본 문장")
print(text)
print()

sentences = sent_tokenize(text)

print("1. 문장 토큰화")
for i, sentence in enumerate(sentences, start=1):
    print(f"문장 {i} : {sentence}")

print()
print("2. 단어 토큰화")
words = word_tokenize(text)
print(words)

print()
print("3. Stemming (어간 추출)")
stemmer = PorterStemmer()
stems = [stemmer.stem(word) for word in words]

for original, stem in zip(words, stems):
    print(f"{original:15} -> {stem}")

print()
print("4. Subword Tokenization")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
subwords = tokenizer.tokenize(text)
print(subwords)

print()
print("5. BERT 입력 ")
encoded = tokenizer(text)
print("input_ids :")
print(encoded["input_ids"])
print("\nattention_mask : ")
print(encoded["attention_mask"])

print("6. 토큰과 숫자 ID를 같이 확인")
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print()
for token, token_id in zip(tokens, encoded["input_ids"]):
    print(f"{token:15} -> {token_id}")

원본 문장
I really enjoyed this movie. The actors were amazing, and the story was surprisingly touching! I would definitely recommend it to my friends.

1. 문장 토큰화
문장 1 : I really enjoyed this movie.
문장 2 : The actors were amazing, and the story was surprisingly touching!
문장 3 : I would definitely recommend it to my friends.

2. 단어 토큰화
['I', 'really', 'enjoyed', 'this', 'movie', '.', 'The', 'actors', 'were', 'amazing', ',', 'and', 'the', 'story', 'was', 'surprisingly', 'touching', '!', 'I', 'would', 'definitely', 'recommend', 'it', 'to', 'my', 'friends', '.']

3. Stemming (어간 추출)
I               -> i
really          -> realli
enjoyed         -> enjoy
this            -> thi
movie           -> movi
.               -> .
The             -> the
actors          -> actor
were            -> were
amazing         -> amaz
,               -> ,
and             -> and
the             -> the
story           -> stori
was             -> wa
surprisingly    -> surprisingli
touching        -> touch
!          

In [ ]:
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import CountVectorizer

print("1. 사용할 문장")

texts = ["I love this movie.", "This movie is really good.", "I hate this movie."]

print("원본 문장")

for text in texts:
    print(text)

print("2. Tokenizer")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("3. Tokenization")

for text in texts:
    tokens = tokenizer.tokenize(text)
    print(f"\n 원문 : {text}")
    print(f"\n 토큰 : {tokens}")

print()
print("4. Token을 숫자 ID로 변환")

for text in texts:
    encoded = tokenizer(text)

    print(f"\n원문 : {text}")
    print(f"input_ids : {encoded['input_ids']}")


print("5. Token과 Token ID를 함께 확인")
text = texts[0]

encoded = tokenizer(text)

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

token_ids = encoded["input_ids"]

for token, token_id in zip(token, token_ids):
    print(f"{token:15} -> {token_id}")

1. 사용할 문장
원본 문장
I love this movie.
This movie is really good.
I hate this movie.
2. Tokenizer
3. Tokenization

 원문 : I love this movie.

 토큰 : ['i', 'love', 'this', 'movie', '.']

 원문 : This movie is really good.

 토큰 : ['this', 'movie', 'is', 'really', 'good', '.']

 원문 : I hate this movie.

 토큰 : ['i', 'hate', 'this', 'movie', '.']

4. Token을 숫자 ID로 변환

원문 : I love this movie.
input_ids : [101, 1045, 2293, 2023, 3185, 1012, 102]

원문 : This movie is really good.
input_ids : [101, 2023, 3185, 2003, 2428, 2204, 1012, 102]

원문 : I hate this movie.
input_ids : [101, 1045, 5223, 2023, 3185, 1012, 102]


In [14]:
import torch
import torch.nn.functional as F
from collections import Counter

raw_sentences = [
    "이 영화 정말 재미있다",
    "이 영화 정말 감동적이다",
    "배우들의 연기가 정말 좋다",
    "스토리가 재미있고 배우들의 연기도 좋다",
]

tokenized_sentences = [sent.split() for sent in raw_sentences]
print("tokenized_sentences : ", tokenized_sentences)

all_tokens = [token for sent in tokenized_sentences for token in sent]
print()
print("all_tokens : ", all_tokens)

vocab_counts = Counter(all_tokens)
print()
print("vocab_counts : ", vocab_counts)

word_to_ids = {"<PAD>": 0, "<UNK>": 1}
for word, _ in vocab_counts.most_common():
    if word not in word_to_ids:
        word_to_ids[word] = len(word_to_ids)
print()
print("word_to_ids : ", word_to_ids)

vocab_size = len(word_to_ids)
print()
print("vocab_size : ", vocab_size)

integer_encoded_sentences = []
for sent in tokenized_sentences:
    int_seq = [word_to_ids.get(token, word_to_ids["<UNK>"]) for token in sent]
    integer_encoded_sentences.append(int_seq)
print()
print("integer_encoded_sentences : ", integer_encoded_sentences)

for i, (origin_sent, int_seq) in enumerate(
    zip(raw_sentences, integer_encoded_sentences)
):
    tensor_seq = torch.tensor(int_seq, dtype=torch.long)
    one_hot_tensor = F.one_hot(tensor_seq, num_classes=vocab_size)

    print()
    print("tensor_seq : ", tensor_seq)
    print("one_hot_tensor : ", one_hot_tensor)

tokenized_sentences :  [['이', '영화', '정말', '재미있다'], ['이', '영화', '정말', '감동적이다'], ['배우들의', '연기가', '정말', '좋다'], ['스토리가', '재미있고', '배우들의', '연기도', '좋다']]

all_tokens :  ['이', '영화', '정말', '재미있다', '이', '영화', '정말', '감동적이다', '배우들의', '연기가', '정말', '좋다', '스토리가', '재미있고', '배우들의', '연기도', '좋다']

vocab_counts :  Counter({'정말': 3, '이': 2, '영화': 2, '배우들의': 2, '좋다': 2, '재미있다': 1, '감동적이다': 1, '연기가': 1, '스토리가': 1, '재미있고': 1, '연기도': 1})

word_to_ids :  {'<PAD>': 0, '<UNK>': 1, '정말': 2, '이': 3, '영화': 4, '배우들의': 5, '좋다': 6, '재미있다': 7, '감동적이다': 8, '연기가': 9, '스토리가': 10, '재미있고': 11, '연기도': 12}

vocab_size :  13

integer_encoded_sentences :  [[3, 4, 2, 7], [3, 4, 2, 8], [5, 9, 2, 6], [10, 11, 5, 12, 6]]

tensor_seq :  tensor([3, 4, 2, 7])
one_hot_tensor :  tensor([[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]])

tensor_seq :  tensor([3, 4, 2, 8])
one_hot_tensor :  tensor([[0, 0